### **The purpose of this notebook is to visualize the spatial distribution of LEiDA states.**

In [ ]:
# Import libraries for data handling, surface plotting, and file operations
import pickle
import pandas as pd
import numpy as np
from enigmatoolbox.utils.parcellation import parcel_to_surface
from enigmatoolbox.plotting import plot_cortical
import re,os
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# Plot centroid values on cortical surface and save as screenshot
def fun_plot_surf(values,filename):
    # Parcel ROI values to fsa5 surface vertices
    values_fsa5 = parcel_to_surface(values, f'schaefer_{schaefer}_fsa5')
    plot_cortical(array_name=values_fsa5,  # Input surface data
                  surface_name="fsa5",      # Surface template
                  size=(1200, 300),         # Output image size
                  cmap='RdBu_r',            # Colormap
                  color_bar=True,           # Show color bar
                  color_range=(-0.1, 0.1),  # Color scale range
                  screenshot=True,          # Save as image file
                  filename=filename,        # Output file path
                  background=(1, 1, 1),     # White background
                  transparent_bg=False,     # Opaque background
                  scale=(3,3),              # Cortical surface scale
                  dpi=300,                  # Image resolution
             )

# Load K-means centroids from a LEiDA result directory
def read_centroids(schaefer,k = 3):
    result_path = f"./LEiDA_results_DMN-{schaefer}/"
    os.makedirs(f"{result_path}/state_plotting", exist_ok = True)
    model_path = f"{result_path}/clustering/models/model_k_{k}.pkl"
    with open(model_path, 'rb') as f:
        kmeans_model = pickle.load(f)
    centroids = kmeans_model.cluster_centers_
    result = []
    for i in range(k):
        result.append(centroids[i])
    return result


In [ ]:
# Set atlas resolution and select Limbic + Default network ROIs
schaefer = 100
atlas = pd.read_csv(f"/data/dy/atlas/upgrade/Schaefer{schaefer}x7_MNI.csv")
chosen_network = ['Limbic',"Default"]
rois = []
for rsn in chosen_network:
    rois += atlas[atlas["ICN"] == rsn]["Name"].tolist()


In [ ]:
# Load K-means centroids (K=5) from a specific LEiDA result directory
result_path = f"./LEiDA_results1"
os.makedirs(f"{result_path}/state_plotting", exist_ok = True)
model_path = f"{result_path}/clustering/models/model_k_5.pkl"
with open(model_path, 'rb') as f:
    kmeans_model = pickle.load(f)
centroids = kmeans_model.cluster_centers_


In [ ]:
# Plot each of the 5 states as a cortical surface map
for i in range(5):
    # Fill full atlas with centroid values (only Limbic/Default ROIs have data, rest are 0)
    values = pd.DataFrame(np.zeros(schaefer), index = atlas["Name"].tolist())
    values.loc[rois,0] = centroids[i,:]
    values = np.array(values).flatten()
    fun_plot_surf(values,f"{result_path}/state_plotting/k_5_state{i+1}.png")
